# Ablation 3 – SE After Stage 2

SE-Block inserted after Swin Stage 2 (C=256). Mid-level features; expected: small accuracy gain over Stage-1 variant.

**Part of the SE placement ablation study.**  
All notebooks share identical data pipelines, training loops, and hyperparameters — only the SE placement differs.

In [ ]:
!pip install timm torch torchvision scikit-learn matplotlib seaborn h5py

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, precision_recall_fscore_support,
                              roc_auc_score)
import timm
import warnings
warnings.filterwarnings('ignore')

In [ ]:
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(42)

In [ ]:
train_dir = 'NIAD-LL/Train'
val_dir   = 'NIAD-LL/Val'
test_dir  = 'NIAD-LL/Test'

## Data Pipeline

In [ ]:
class AnomalyDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir   = root_dir
        self.transform  = transform
        self.images, self.labels = [], []
        self.class_names = ['Normal', 'Anomalous']
        for label, folder in enumerate(['Normal', 'Anomalous']):
            d = os.path.join(root_dir, folder)
            if os.path.exists(d):
                for f in os.listdir(d):
                    if f.lower().endswith(('.png','.jpg','.jpeg','.bmp','.tiff')):
                        self.images.append(os.path.join(d, f))
                        self.labels.append(label)
        print(f"Loaded {len(self.images)} images — Normal:{self.labels.count(0)}  Anomalous:{self.labels.count(1)}")
    def __len__(self): return len(self.images)
    def __getitem__(self, idx):
        try:   img = Image.open(self.images[idx]).convert('RGB')
        except: img = Image.new('RGB',(224,224))
        if self.transform: img = self.transform(img)
        return img, self.labels[idx]

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomVerticalFlip(0.2),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2,contrast=0.2,saturation=0.2,hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1,0.1)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])
val_test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

In [ ]:
train_dataset = AnomalyDataset(train_dir, transform=train_transform)
val_dataset   = AnomalyDataset(val_dir,   transform=val_test_transform)
test_dataset  = AnomalyDataset(test_dir,  transform=val_test_transform)

batch_size  = 32
num_workers = 2
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  num_workers=num_workers, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
print(f"Train batches:{len(train_loader)}  Val:{len(val_loader)}  Test:{len(test_loader)}")

## Model Definition

In [ ]:
# ── Ablation: SE inserted after Swin Stage 2 ──────────────────────────
# after Stage 2 (second transformer block, C=256)
import timm, torch, torch.nn as nn

class ChannelAttention(nn.Module):
    """SE-Block: GAP → MLP(Linear-ReLU-Linear-Sigmoid) → scale."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        mid = max(1, channels // reduction)
        self.att = nn.Sequential(
            nn.Linear(channels, mid, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.att(x)


class SwinSEStage2(nn.Module):
    """SE-Block inserted after Stage 2 (second transformer block, C=256).
    Pipeline:
      Swin Stage 1-2 → intermediate feature (B,H,W,C) captured via hook
      → SE on channel dim → remaining stages → GAP → Classifier
    Note: The SE block is injected *between* Swin stages using a forward hook
    so the backbone's weight structure remains identical across all ablations.
    """
    def __init__(self, num_classes=2, pretrained=True, reduction=16):
        super().__init__()
        self.swin = timm.create_model('swin_base_patch4_window7_224',
                                       pretrained=pretrained, num_classes=0)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        feature_channels = 1024          # final backbone output
        se_channels      = 256   # output channels of Stage 2

        # SE block operating on Stage-2 feature maps
        self.se_stage2 = ChannelAttention(se_channels, reduction)

        self.classifier = nn.Sequential(
            nn.Linear(feature_channels, 512), nn.LayerNorm(512),
            nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(512, 256), nn.LayerNorm(256),
            nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

        # ── Register forward hook on swin.layers[1] ──────────────
        # The hook intercepts the output of Stage 2, applies SE, and
        # writes the rescaled tensor back so downstream stages see it.
        self._hook_handle = None
        self._register_se_hook()

    def _register_se_hook(self):
        def hook_fn(module, input, output):
            # output shape from Swin stage: (B, H, W, C) or (B, seq, C)
            if output.dim() == 4:
                B, H, W, C = output.shape
                flat = output.reshape(B, H*W, C).mean(dim=1)   # (B,C) for SE
                scale = self.se_stage2(flat)              # (B,C) weights
                # Broadcast scale back to (B,H,W,C) and rescale
                return output * scale.unsqueeze(1).unsqueeze(1)
            elif output.dim() == 3:
                B, S, C = output.shape
                flat  = output.mean(dim=1)                      # (B,C)
                scale = self.se_stage2(flat)              # (B,C)
                return output * scale.unsqueeze(1)
            return output

        self._hook_handle = self.swin.layers[1].register_forward_hook(hook_fn)

    def forward(self, x):
        feat = self.swin(x)   # hook fires automatically inside here
        if feat.dim() == 3:   feat = feat.mean(dim=1)
        elif feat.dim() == 4: feat = self.global_pool(feat).flatten(1)
        return self.classifier(feat)

    def __del__(self):
        if self._hook_handle is not None:
            self._hook_handle.remove()


## Initialise Model, Loss & Optimiser

In [ ]:
# ── Device & model init ───────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

model = SwinSEStage2(num_classes=2, pretrained=True).to(device)

# Freeze all, unfreeze last 2 Swin stages + new heads
for param in model.parameters():
    param.requires_grad = False
for name, param in model.swin.named_parameters():
    if name.startswith('layers.2') or name.startswith('layers.3'):
        param.requires_grad = True
for name, param in model.named_parameters():
    if any(k in name for k in ['channel_attention','se_stage','se1','se2',
                                'se3','se4','classifier']):
        param.requires_grad = True

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total:,}  |  Trainable: {trainable:,}  ({100*trainable/total:.1f}%)")

# Loss
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

# Param groups with different learning rates
swin_params = [p for n,p in model.named_parameters()
               if p.requires_grad and ('swin.layers.2' in n or 'swin.layers.3' in n)]
head_params = [p for n,p in model.named_parameters()
               if p.requires_grad and not ('swin.layers.2' in n or 'swin.layers.3' in n)]

optimizer = optim.AdamW([
    {'params': swin_params, 'lr': 1e-5, 'weight_decay': 1e-4},
    {'params': head_params, 'lr': 1e-4, 'weight_decay': 1e-4},
])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-7)
print(f"Optimiser ready — swin fine-tune: {len(swin_params)} tensors  head: {len(head_params)} tensors")

## Training & Evaluation

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = correct = total = 0
    pbar = tqdm(dataloader, desc='Training')
    for inputs, labels in pbar:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total   += labels.size(0)
        correct += (predicted == labels).sum().item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100*correct/total:.2f}%'})
    return running_loss/len(dataloader), 100*correct/total

In [ ]:
def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = correct = total = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        pbar = tqdm(dataloader, desc='Validation')
        for inputs, labels in pbar:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total   += labels.size(0)
            correct += (predicted == labels).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100*correct/total:.2f}%'})
    return running_loss/len(dataloader), 100*correct/total, all_preds, all_labels

In [ ]:
num_epochs  = 50
best_val_acc = 0.0
train_losses, train_accs, val_losses, val_accs = [], [], [], []

for epoch in range(num_epochs):
    print(f'\nEpoch [{epoch+1}/{num_epochs}]')
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc, _, _ = validate(model, val_loader, criterion, device)
    scheduler.step()
    train_losses.append(train_loss); train_accs.append(train_acc)
    val_losses.append(val_loss);     val_accs.append(val_acc)
    print(f'Train Loss:{train_loss:.4f} Acc:{train_acc:.2f}% | Val Loss:{val_loss:.4f} Acc:{val_acc:.2f}%')
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')
        print(f'  ✓ Best model saved (val_acc={best_val_acc:.2f}%)')
print(f'\nTraining complete. Best Val Acc: {best_val_acc:.2f}%')

In [ ]:
# ── Final evaluation on test set ──────────────────────────────────────────
model.load_state_dict(torch.load('best_model.pth'))
_, test_acc, test_preds, test_labels_list = validate(model, test_loader, criterion, device)

print(f'\nTest Accuracy: {test_acc:.4f}%')
print('\nClassification Report:')
print(classification_report(test_labels_list, test_preds,
      target_names=['Normal','Anomalous'], digits=4))

cm = confusion_matrix(test_labels_list, test_preds)
fig, axes = plt.subplots(1,2,figsize=(14,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal','Anomalous'],
            yticklabels=['Normal','Anomalous'], ax=axes[0])
axes[0].set_title(f'Confusion Matrix  (Acc={test_acc:.2f}%)')
cm_n = cm.astype(float)/cm.sum(axis=1,keepdims=True)
sns.heatmap(cm_n, annot=True, fmt='.3f', cmap='Blues',
            xticklabels=['Normal','Anomalous'],
            yticklabels=['Normal','Anomalous'], ax=axes[1])
axes[1].set_title('Normalised Confusion Matrix')
plt.tight_layout(); plt.show()

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1,2,figsize=(14,5))
axes[0].plot(train_losses, label='Train'); axes[0].plot(val_losses, label='Val')
axes[0].set_title('Loss Curves'); axes[0].legend()
axes[1].plot(train_accs, label='Train'); axes[1].plot(val_accs, label='Val')
axes[1].set_title('Accuracy Curves'); axes[1].legend()
plt.tight_layout(); plt.show()